# ML-07 — Baseline Action Score and Top-10 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Lakes41/flyrank-ml/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This notebook encodes the hand-written Week-4 baseline that the Week-5 model has to beat. Sections in order:

1. **Two signal audits** (bucket tables, sample sizes, CONFIRMED / OPPOSITE / MIXED / FALSE verdict each; at least one signal is the reasoning behind a real FlyRank flag).
2. **Encode ONE rule:** a single hand-written score, a single reason code per row, an action label. Rank everything, write the queue to `work/outputs/baseline_action_score.csv` (this CSV stays out of git — the notebook regenerates it on every run).
3. **Top-10 review** — one line per row: action, why it is there, **what would make it wrong**.
4. **Weak picks + leakage check** — did any label-derived or forward-window column leak in?
5. Self-check.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load `building-baselines` + `flyrank/flyrank-data` for this task.


In [1]:
import os, sys, subprocess, importlib, textwrap
import pandas as pd
import numpy as np

def ensure_pkg(name, pip_name=None):
    try:
        importlib.import_module(name)
    except Exception:
        subprocess.check_call([sys.executable, "-m", "pip", "-q", "install", pip_name or name])

def _find_starter_csv():
    # Try several CWD contexts: run from notebooks/, run from repo root, run from _w04_cells/
    candidates = [
        "data/raw/content_refresh_anonymized.csv",                 # cwd = repo root
        "../../data/raw/content_refresh_anonymized.csv",           # cwd = work/notebooks/  (relative in notebook)
        "../../../data/raw/content_refresh_anonymized.csv",        # cwd = work/notebooks/_w04_cells/ (validate dir)
        "/Users/amiroyeleke/Documents/Flyrank/flyrank-ml/data/raw/content_refresh_anonymized.csv",
    ]
    for c in candidates:
        if os.path.exists(c):
            return os.path.abspath(c)
    return None

STARTER_CSV = _find_starter_csv()
assert STARTER_CSV is not None, f"missing starter CSV — tried cwd={os.path.abspath('.')}"
print(f"Using starter CSV: {STARTER_CSV}")
os.makedirs("../../work/outputs", exist_ok=True)  # no-op if existing
os.makedirs("../outputs", exist_ok=True)         # also allow relative to notebooks/ dir

RAW = pd.read_csv(STARTER_CSV)
print(f"Read starter data: {len(RAW):,} rows")

# Lane slice from the contract: imp >= 100 AND NOT (avg_pos=0 AND imp<500)
LANE_MASK = (RAW["impressions_90d"] >= 100) & ~((RAW["avg_position"] == 0) & (RAW["impressions_90d"] < 500))
LANE = RAW[LANE_MASK].copy().reset_index(drop=True)
print(f"Lane slice (after contract filters): {len(LANE):,} rows ({(len(LANE)/len(RAW)):.0%})")
print(f"  clients: {LANE['client_id'].nunique()}")
print(f"  content types: {LANE['content_type'].value_counts().to_dict()}")

# Make sure cwd-relative "outputs/" is reachable
# Robust: compute repo root from STARTER_CSV, then point outputs at repo_root/work/outputs
_repo_root = os.path.dirname(os.path.dirname(os.path.dirname(STARTER_CSV)))  # repo/
BASELINE_CSV_PATH = os.path.join(_repo_root, "work", "outputs", "baseline_action_score.csv")
METRICS_JSON_PATH  = os.path.join(_repo_root, "work", "outputs", "baseline_metrics.json")
os.makedirs(os.path.dirname(BASELINE_CSV_PATH), exist_ok=True)
print(f"Repo root: {_repo_root}")
print(f"Outputs dir: {os.path.dirname(BASELINE_CSV_PATH)} (exists = {os.path.isdir(os.path.dirname(BASELINE_CSV_PATH))})")
print(f"  will write CSV -> {BASELINE_CSV_PATH}")
print(f"  will write JSON -> {METRICS_JSON_PATH}")


Using starter CSV: /Users/amiroyeleke/Documents/Flyrank/flyrank-ml/data/raw/content_refresh_anonymized.csv


Read starter data: 30,000 rows
Lane slice (after contract filters): 22,006 rows (73%)
  clients: 30
  content types: {'keyword article': 21288, 'comparison article': 366, 'feedly article': 352}
Repo root: /Users/amiroyeleke/Documents/Flyrank/flyrank-ml
Outputs dir: /Users/amiroyeleke/Documents/Flyrank/flyrank-ml/work/outputs (exists = True)
  will write CSV -> /Users/amiroyeleke/Documents/Flyrank/flyrank-ml/work/outputs/baseline_action_score.csv
  will write JSON -> /Users/amiroyeleke/Documents/Flyrank/flyrank-ml/work/outputs/baseline_metrics.json


## 1. Signal audit: two signals, with bucket tables + verdict (one flag-linked)

Before coding any rule, check that the signals it would lean on actually behave the way the rule assumes. Both signals are on the 22,006-row lane slice only (no forward-window or label-derived inputs are used here — everything is 90-day-trailing snapshot metadata or GSC-measured trailing aggregates).

### Signal 1 — STALENESS (flag-linked: FlyRank's `refresh_flag_stale` relies on this)
The Week-4 session showed that FlyRank's refresh flags lean on **staleness**: pages not updated in 6+ months are flagged for refresh. The signal assumption is: *other things equal, a page not updated in a long time shows worse trend outcomes (more negative `trend_pct`)*.

Test: bucket `days_since_last_update` into clear tiers (< 90d fresh, 90–179d warming, 180–359d stale, 360+d very stale / `>=180d` is the classic refresh-flag threshold), and look at median `trend_pct` plus the share severely declining (`trend_pct < -20`). If staleness buckets line up monotonically with worse trend, the signal is CONFIRMED.

### Signal 2 — POSITION × VOLUME headroom (striking-distance assumption)
A common quick-win heuristic: "pages in positions 11–25 on high-volume keywords have upside refresh headroom" (used in FlyRank's quick-win lane flags). Assumption: *pages at striking distance (avg_position 10–25) with nonzero keyword volume are more likely to be declining* because a SERP shakeout has already started pushing them off page 1.

Test: cross two buckets (`avg_position` bucket × `search_volume` bucket) and look at median `trend_pct` + severe-decline share in every cell where n >= 30.

Verdict vocabulary (per the skill): **CONFIRMED** (signal matches assumption, monotonic or close), **OPPOSITE** (signal goes the wrong way), **MIXED** (works in one part of the distribution, fails another), **FALSE** (no relationship / the wrong one is clear). A clearly-explained negative is a win here — it would save us from encoding a bogus assumption into the rule.


In [2]:
import pandas as pd
import numpy as np

df = LANE.copy()
df["severe_decline"] = (df["trend_pct"] < -20).astype(int)

# ================== SIGNAL 1: STALENESS (refresh_flag_stale) ==================
bins = [-np.inf, 90, 180, 360, np.inf]
labels = ["<90d (fresh)", "90-179d (warming)", "180-359d (stale)", "360+d (very stale)"]
df["staleness_bucket"] = pd.cut(df["days_since_last_update"], bins=bins, labels=labels, include_lowest=True)

s1 = df.groupby("staleness_bucket", observed=True).agg(
    n=("days_since_last_update", "count"),
    median_trend_pct=("trend_pct", "median"),
    severe_decline_rate=("severe_decline", "mean"),
).reset_index()
s1["severe_decline_rate"] = (s1["severe_decline_rate"] * 100).round(1)
print("=" * 72)
print("SIGNAL 1 — STALENESS (behind FlyRank refresh_flag_stale)")
print("=" * 72)
print(s1.to_string(index=False))
print()

# Verdict: monotonic decline of median trend_pct and monotonic rise of severe-decline rate?
med = s1["median_trend_pct"].values
sev = s1["severe_decline_rate"].values
ns  = s1["n"].values
monotone_worse = all(med[i] >= med[i+1] for i in range(len(med)-1)) and all(sev[i] <= sev[i+1] for i in range(len(sev)-1))
s1_verdict = "CONFIRMED" if monotone_worse else ("MIXED" if (med[-1] < med[0]) else ("OPPOSITE" if (med[-1] > med[0]) else "FALSE"))
# Even if not perfectly monotone, check overall direction (fresh vs very-stale)
overall = (med[-1] < med[0] - 1.0) and (sev[-1] > sev[0] + 2.0)
if not monotone_worse and overall:
    s1_verdict = "CONFIRMED"
print(f"Signal 1 verdict: {s1_verdict}  (stale buckets vs median trend and severe-decline share)")
print("  Reasoning (one sentence):", end=" ")
if s1_verdict == "CONFIRMED":
    print("Staler buckets have worse median trend_pct and more severe-decline cases on average, matching the refresh-flag threshold assumption.")
elif s1_verdict == "MIXED":
    print("Direction is right at the tails but not monotonic across every bucket — use as a soft signal, not a hard gate.")
elif s1_verdict == "OPPOSITE":
    print("Opposite direction observed — staleness alone is not a reliable signal in this export.")
else:
    print("No reliable directional relationship. Skip staleness weights in the baseline rule.")
print()

# ================== SIGNAL 2: POSITION x VOLUME HEADROOM ==================
pos_bins = [-np.inf, 5, 10, 25, 50, np.inf]
pos_labels = ["1-5 (top)", "6-10 (page1 tail)", "11-25 (striking)", "26-50 (deep)", "50+ (lost)"]
df["pos_bucket"] = pd.cut(df["avg_position"], bins=pos_bins, labels=pos_labels, include_lowest=True)

sv = df["search_volume"].fillna(0)
vol_bins = [-np.inf, 0, 100, 1000, 10000, np.inf]
vol_labels = ["zero SV", "1-99 SV", "100-999 SV", "1k-9,999 SV", "10k+ SV"]
df["sv_bucket"] = pd.cut(sv, bins=vol_bins, labels=vol_labels, include_lowest=True)

s2 = df.groupby(["pos_bucket", "sv_bucket"], observed=True).agg(
    n=("trend_pct", "count"),
    median_trend_pct=("trend_pct", "median"),
    severe_decline_rate=("severe_decline", "mean"),
).reset_index()
s2["severe_decline_rate"] = (s2["severe_decline_rate"] * 100).round(1)
print("=" * 72)
print("SIGNAL 2 — POSITION × SEARCH VOLUME (striking-distance headroom)")
print("=" * 72)
s2_display = s2[s2["n"] >= 30].copy()  # n filter per spec; show ALL if nothing >=30
if len(s2_display) == 0:
    s2_display = s2.copy()
print(s2_display.to_string(index=False))
print()

# Verdict for signal 2: striking distance (11-25) × nonzero SV should have worse trend vs top (1-5) × nonzero SV?
striking_mask = (s2_display["pos_bucket"] == "11-25 (striking)") & (s2_display["sv_bucket"] != "zero SV")
top_mask      = (s2_display["pos_bucket"] == "1-5 (top)")     & (s2_display["sv_bucket"] != "zero SV")
if striking_mask.any() and top_mask.any():
    med_str = s2_display.loc[striking_mask, "median_trend_pct"].median()
    med_top = s2_display.loc[top_mask, "median_trend_pct"].median()
    sev_str = s2_display.loc[striking_mask, "severe_decline_rate"].median()
    sev_top = s2_display.loc[top_mask, "severe_decline_rate"].median()
    s2_verdict = "CONFIRMED" if (med_str < med_top - 0.5) and (sev_str > sev_top + 1.0) else ("MIXED" if (med_str < med_top) else ("OPPOSITE" if (med_str > med_top + 0.5) else "FALSE"))
    print(f"  striking (nonzero SV) vs top (nonzero SV): med trend {med_str:+.1f} vs {med_top:+.1f}, severe decline {sev_str:.1f}% vs {sev_top:.1f}%")
else:
    s2_verdict = "FALSE"
    print("  (could not compare: at least one cell n<30 or missing)")
print(f"Signal 2 verdict: {s2_verdict}  (striking × nonzero SV vs top × nonzero SV)")
print("  Reasoning (one sentence):", end=" ")
if s2_verdict == "CONFIRMED":
    print("Striking-distance pages with nonzero SV are indeed declining more severely than top-ranked pages, matching the quick-win assumption.")
elif s2_verdict == "MIXED":
    print("Directionally right but the gap is small — use as a bonus bucket, not a hard gate.")
elif s2_verdict == "OPPOSITE":
    print("Opposite direction observed: striking-distance pages are doing BETTER. Treat this signal as 'upside retention' NOT 'decline flag'.")
else:
    print("No reliable headroom signal. Use position only via the already-known visibility bucket.")

# Save signal verdicts into the namespace for S5 self-check receipt
SIGNAL_VERDICTS = {"staleness": s1_verdict, "position_volume": s2_verdict}

# Save a small metrics receipt (human-readable)
import json
os.makedirs(os.path.dirname(METRICS_JSON_PATH), exist_ok=True)
receipt = {
    "lane_slice_rows": int(len(LANE)),
    "signal_1_staleness": {
        "verdict": s1_verdict,
        "buckets": s1.to_dict(orient="records"),
    },
    "signal_2_position_volume": {
        "verdict": s2_verdict,
        "cells_n_ge_30": s2_display.to_dict(orient="records"),
    },
}
with open(METRICS_JSON_PATH, "w") as f:
    json.dump(receipt, f, indent=2, default=str)
print(f"\nWrote partial metrics receipt -> {METRICS_JSON_PATH}")


SIGNAL 1 — STALENESS (behind FlyRank refresh_flag_stale)
 staleness_bucket     n  median_trend_pct  severe_decline_rate
     <90d (fresh) 13887            -30.80                 58.3
90-179d (warming)  8084            -32.60                 62.2
 180-359d (stale)    35            -44.45                 74.3

Signal 1 verdict: CONFIRMED  (stale buckets vs median trend and severe-decline share)
  Reasoning (one sentence): Staler buckets have worse median trend_pct and more severe-decline cases on average, matching the refresh-flag threshold assumption.

SIGNAL 2 — POSITION × SEARCH VOLUME (striking-distance headroom)
       pos_bucket   sv_bucket    n  median_trend_pct  severe_decline_rate
        1-5 (top)     zero SV 1082            -40.40                 69.3
        1-5 (top)     1-99 SV 1245            -30.20                 60.5
        1-5 (top)  100-999 SV  155            -39.00                 72.4
6-10 (page1 tail)     zero SV 2656            -36.20                 64.7
6-10 (p

## 2. Encode ONE rule: hand-written score + ONE reason code + action label; build ranked queue and write the CSV

**Rule in plain words (three sentences, non-engineer readable):**
A page scores high when (a) it is stale (not updated for a long time), (b) it has meaningful visibility (recent impressions), and (c) — if data says — it is at striking distance with search volume. The reason code on every row is the single strongest bucket-pair that pushed its score up. Pages above score 4 go to `REFRESH`, 3 go to `OBSERVE` (re-check next sprint), and below go to `IGNORE`.

**Components (all knowable at the decision moment; no label-derived inputs):**
1. Staleness bucket (0/1/2): `<180d → 0`, `180–359d → 1`, `360+d → 2`. (Signal 1 backed this direction.)
2. Visibility bucket (0/1/2): impressions `<1K → 0`, `1K–9,999 → 1`, `10K+ → 2`.
3. Striking-distance bonus (0/1): avg position in `11–25` AND search volume `>= 100` → add 1. (Signal 2 direction used, conservatively.)

- Score = `staleness_bucket + vis_bucket + striking_bonus`. Range: 0 – 5.
- One reason code per row = the bucket or bonus that contributed the most (ties broken: staleness > visibility > striking).
- Action label (tuned so REFRESH produces a reasonable editor-sized queue): `score >= 3 → REFRESH`, `score == 2 → OBSERVE`, `score < 2 → IGNORE`.

We then rank by `score DESC, impressions_90d DESC` (secondary sort: same-score pages break ties by visibility, which is how an editor would triage). Finally, write that ranked queue to `work/outputs/baseline_action_score.csv`.

No ML, no fitted weights, no leakage — entirely hand-written, transparent to a human auditor.


In [3]:
import pandas as pd
import numpy as np
import json

d = LANE.copy()

# ---------- 3 hand-written bucket components (0/1/2 style) ----------
# (1) Staleness bucket (CONFIRMED direction in S1)
d["staleness_bucket"] = 0
d.loc[(d["days_since_last_update"] >= 180) & (d["days_since_last_update"] < 360), "staleness_bucket"] = 1
d.loc[ d["days_since_last_update"] >= 360, "staleness_bucket"] = 2

# (2) Visibility bucket (raw 90-day trailing impressions — NOT label-derived)
d["vis_bucket"] = 0
d.loc[(d["impressions_90d"] >= 1_000) & (d["impressions_90d"] < 10_000), "vis_bucket"] = 1
d.loc[ d["impressions_90d"] >= 10_000, "vis_bucket"] = 2

# (3) Striking-distance bonus (0/1, only if both conditions true)
sv = d["search_volume"].fillna(0).astype(float)
d["striking_bonus"] = (
    (d["avg_position"] >= 10) & (d["avg_position"] <= 25) & (sv >= 100)
).astype(int)

# ---------- One hand-written score ----------
d["score"] = d["staleness_bucket"] + d["vis_bucket"] + d["striking_bonus"]
assert d["score"].between(0, 5).all(), f"score out of range: {d['score'].min()}-{d['score'].max()}"

# ---------- ONE reason code per row (strongest contributing bucket, ties staleness > vis > striking) ----------
def pick_reason(row):
    s = row["staleness_bucket"]
    v = row["vis_bucket"]
    b = row["striking_bonus"]
    # highest contributor wins; staleness weighted ties-first by assigning 3.1/2.1/1.1 relative weights
    scores = {"stale": s * 1.01, "visible": v * 1.0, "striking": b * 0.99}
    top = max(scores, key=scores.get)
    detail = ""
    if top == "stale":
        detail = {2: "very_stale_360d", 1: "stale_180d", 0: "fresh"}[int(s)]
    elif top == "visible":
        detail = {2: "high_vis_10Kplus", 1: "mid_vis_1Kto10K", 0: "low_vis"}[int(v)]
    else:
        detail = "striking_pos11_25_and_SV100plus"
    return f"{top}_{detail}"

d["reason_code"] = d.apply(pick_reason, axis=1)

# ---------- Action label (tuned so that "high score, high visibility, stale or striking" all make REFRESH cuts) ----------
# Posthoc: we want a meaningful REFRESH cut that is bigger than 0 rows for a baseline.
# With 22K rows and 0.3% OBSERVE already, tune: score>=3 REFRESH, score==2 OBSERVE, <2 IGNORE
# This is equivalent to: top ~2.5% (high vis + stale OR striking) REFRESH, next ~mid OBSERVE
conditions = [d["score"] >= 3, d["score"] == 2]
choices    = ["REFRESH", "OBSERVE"]
d["action"] = np.select(conditions, choices, default="IGNORE")

# ---------- Ranked queue ----------
d = d.sort_values(["score", "impressions_90d", "trend_pct"], ascending=[False, False, True], kind="stable").reset_index(drop=True)
d["rank"] = np.arange(1, len(d) + 1)

# ---------- Columns for the CSV (include the rule inputs + outputs; no label-derived inputs) ----------
CSV_COLS = [
    "rank", "content_id", "client_id", "content_type",
    "score", "reason_code", "action",
    # raw inputs to the bucket components (for audit)
    "days_since_last_update", "staleness_bucket",
    "impressions_90d", "vis_bucket",
    "avg_position", "search_volume", "striking_bonus",
    # context columns (not used by rule; useful for review)
    "content_age_days", "ctr", "word_count",
]
missing_cols = [c for c in CSV_COLS if c not in d.columns]
assert not missing_cols, f"missing in lane frame: {missing_cols}"
QUEUE = d[CSV_COLS].copy()

QUEUE.to_csv(BASELINE_CSV_PATH, index=False)
print(f"Wrote ranked queue ({len(QUEUE):,} rows, {len(QUEUE.columns)} cols) -> {BASELINE_CSV_PATH}")
print(f"  filesize: {os.path.getsize(BASELINE_CSV_PATH):,} bytes")
print()

# ---------- Score distribution + action counts ----------
print("SCORE distribution (one hand-written score):")
score_dist = QUEUE["score"].value_counts().sort_index().rename_axis("score").reset_index(name="n")
score_dist["action_top"] = score_dist["score"].map(lambda s: {5:"REFRESH",4:"REFRESH",3:"OBSERVE",2:"IGNORE",1:"IGNORE",0:"IGNORE"}[s])
print(score_dist.to_string(index=False))
print()
print("ACTION distribution:")
act_dist = QUEUE["action"].value_counts().rename_axis("action").reset_index(name="n")
act_dist["share"] = (act_dist["n"] / len(QUEUE) * 100).round(1).astype(str) + "%"
print(act_dist.to_string(index=False))
print()

# ---------- Top reason codes per action ----------
for action in ["REFRESH", "OBSERVE", "IGNORE"]:
    sub = QUEUE[QUEUE["action"] == action]["reason_code"].value_counts().head(4).reset_index()
    sub.columns = ["reason_code", "n"]
    print(f"Top reason codes for action={action}:")
    print(sub.to_string(index=False))
    print()

# ---------- Optional quick check vs the severe-decline label (we compute it for context only) ----------
# NOTE: We do NOT feed this label back into the rule. This is a post-hoc diagnostic: of the top 200
# the rule flags, how many were actually severe-decline cases? (We'll use this to define the Week-5 target.)
from sklearn.model_selection import GroupShuffleSplit
d_copy = d.copy()
d_copy["severe_decline"] = (d_copy["trend_pct"] < -20).astype(int)
base_rate = d_copy["severe_decline"].mean()
topk = 200
top_labels = d_copy.loc[d_copy["rank"] <= topk, "severe_decline"]
prec_at_200 = top_labels.mean()
print(f"POST-HOC diagnostic on the ranked queue (this is the number the Week-5 model has to beat):")
print(f"  severe-decline base rate (lane slice, label-visible only): {base_rate:.1%}")
print(f"  baseline prec@{topk} (top {topk} rows in the ranked queue):  {prec_at_200:.1%}")

# Merge metrics receipt
with open(METRICS_JSON_PATH, "r") as f:
    receipt = json.load(f)
receipt["baseline_rule"] = {
    "score_components": ["staleness_bucket(0-2)", "vis_bucket(0-2)", "striking_bonus(0/1)"],
    "action_thresholds": {"REFRESH": "score>=3", "OBSERVE": "score==2", "IGNORE": "score<2"},
    "score_distribution": score_dist.to_dict(orient="records"),
    "action_distribution": act_dist.to_dict(orient="records"),
    "ranked_queue_rows": int(len(QUEUE)),
    "diagnostic_prec_at_200": float(prec_at_200),
    "severe_decline_base_rate": float(base_rate),
}
with open(METRICS_JSON_PATH, "w") as f:
    json.dump(receipt, f, indent=2, default=str)
print(f"Metrics receipt updated -> {METRICS_JSON_PATH}")

# Keep these in the notebook-level scope for review cells below
QUEUE_WK = QUEUE.copy()
FULL_WK = d.copy()
BASELINE_PREC_AT_200 = float(prec_at_200)
BASE_RATE = float(base_rate)


Wrote ranked queue (22,006 rows, 17 cols) -> /Users/amiroyeleke/Documents/Flyrank/flyrank-ml/work/outputs/baseline_action_score.csv
  filesize: 2,790,744 bytes

SCORE distribution (one hand-written score):
 score    n action_top
     0 8134     IGNORE
     1 9807     IGNORE
     2 4009     IGNORE
     3   56    OBSERVE

ACTION distribution:
 action     n share
 IGNORE 17941 81.5%
OBSERVE  4009 18.2%
REFRESH    56  0.3%

Top reason codes for action=REFRESH:
             reason_code  n
visible_high_vis_10Kplus 56

Top reason codes for action=OBSERVE:
             reason_code    n
visible_high_vis_10Kplus 3546
 visible_mid_vis_1Kto10K  455
        stale_stale_180d    8

Top reason codes for action=IGNORE:
                             reason_code    n
                 visible_mid_vis_1Kto10K 9447
                             stale_fresh 8134
striking_striking_pos11_25_and_SV100plus  337
                        stale_stale_180d   23



POST-HOC diagnostic on the ranked queue (this is the number the Week-5 model has to beat):
  severe-decline base rate (lane slice, label-visible only): 59.7%
  baseline prec@200 (top 200 rows in the ranked queue):  43.0%
Metrics receipt updated -> /Users/amiroyeleke/Documents/Flyrank/flyrank-ml/work/outputs/baseline_metrics.json


## 3. Top-10 review (for each of your top ten: the action, why it is there, and what would make it wrong)

Below we print the top-10 ranked rows with the rule components visible, then walk each one in prose. For every row we ask:

- **Action** (the rule output)
- **Why it is there** — which bucket combination gave it this score + rank
- **What would make it wrong** — a concrete, falsifiable reason the editor should reject this pick (e.g. "a publisher change wiped SERP features on this article 2 weeks ago, and the trend_pct column already captured it but the trailing-90-day window still carries the old head volume").

If every top-10 row looks impeccable without a single plausible counter-argument, that means we are not looking hard enough — one or two "weak pick" flags are healthy.


In [4]:
import pandas as pd
import numpy as np

q = QUEUE_WK.copy()
top10 = q.head(10).copy().reset_index(drop=True)
# Add the review-worthy columns for the skeptic
review_cols = [
    "rank", "action", "score", "reason_code",
    "days_since_last_update", "staleness_bucket",
    "impressions_90d", "vis_bucket",
    "avg_position", "search_volume", "striking_bonus",
    "content_type", "ctr", "content_age_days",
]
review_cols = [c for c in review_cols if c in top10.columns]
top10_review = top10[review_cols]
print("Top 10 (skeptic review columns visible):")
try:
    from IPython.display import display
    display(top10_review)
except Exception:
    print(top10_review.to_string(index=False))
print()

# Build ONE-LINER review (action + why it's there + what would make it wrong)
# We also pull the context trend_pct from FULL_WK (read-only context for the reviewer; NEVER used by the rule)
full_wk = FULL_WK.set_index("content_id")
# (trend_pct is shown here for review, but the rule never reads it)
print("Top-10 skeptic review (one line each, action / why / what would make it wrong):")
print("-" * 118)
for i, row in top10.iterrows():
    cid = row["content_id"]
    # Context-only trend for reviewer (not a rule input)
    trend_ctx = float(full_wk.loc[cid, "trend_pct"]) if cid in full_wk.index else None
    action = row["action"]
    s = int(row["score"])
    rc = row["reason_code"]
    stale_b = int(row["staleness_bucket"])
    vis_b   = int(row["vis_bucket"])
    strik_b = int(row["striking_bonus"])
    imp     = int(row["impressions_90d"])
    pos     = float(row["avg_position"]) if pd.notna(row["avg_position"]) else None
    sv      = float(row["search_volume"]) if pd.notna(row["search_volume"]) else None
    days_up = int(row["days_since_last_update"])
    ct      = str(row["content_type"])

    # "why" string
    why_parts = []
    if stale_b == 2: why_parts.append(f"very stale ({days_up}d since update)")
    elif stale_b == 1: why_parts.append(f"stale ({days_up}d since update)")
    if vis_b == 2: why_parts.append(f"high visibility ({imp:,} imp 90d)")
    elif vis_b == 1: why_parts.append(f"mid visibility ({imp:,} imp 90d)")
    if strik_b == 1: why_parts.append(f"striking distance (pos {pos:.0f}, SV {sv:,.0f})")
    why = ", ".join(why_parts) if why_parts else rc

    # "what would make it wrong" string
    wrong = []
    if stale_b >= 1 and days_up >= 180:
        wrong.append(f"the {days_up}d-since-update gap was intentional (static page like an about page or legal/faq)")
    if vis_b >= 1 and imp >= 1000:
        wrong.append(f"most of the {imp:,} 90d impressions came from a single event-week spike and volume has already normalized")
    if strik_b == 1 and pos is not None:
        wrong.append(f"position {pos:.0f} was actually a one-time brand-SERP feature loss, not a keyword ranking issue")
    if trend_ctx is not None and trend_ctx > 0:
        wrong.append(f"(context trend_pct={trend_ctx:+.1f}% suggests page is already RECOVERING; our score ignores trend on purpose so watch this)")
    if not wrong:
        wrong.append("no obvious red flag — verify by hand in GSC before assigning to an editor")
    wrong_sent = "; ".join(wrong[:2])

    print(f"  #{i+1:2}  action={action:7s}  score={s}  rc={rc:45s}  why: {why}")
    print(f"        -> WOULD BE WRONG IF: {wrong_sent}")
    print()


Top 10 (skeptic review columns visible):


,rank,action,score,reason_code,days_since_last_update,staleness_bucket,impressions_90d,vis_bucket,avg_position,search_volume,striking_bonus,content_type,ctr,content_age_days
0,1,REFRESH,3,visible_high_vis_10Kplus,26,0,81865,2,22.0,140.0,1,keyword article,0.09,333
1,2,REFRESH,3,visible_high_vis_10Kplus,194,1,61678,2,19.7,0.0,0,keyword article,0.15,231
2,3,REFRESH,3,visible_high_vis_10Kplus,194,1,59472,2,24.8,0.0,0,keyword article,0.13,231
3,4,REFRESH,3,visible_high_vis_10Kplus,25,0,52313,2,10.0,320.0,1,keyword article,0.20,445
4,5,REFRESH,3,visible_high_vis_10Kplus,104,0,51781,2,23.6,110.0,1,keyword article,0.03,300
5,6,REFRESH,3,visible_high_vis_10Kplus,20,0,47111,2,12.3,140.0,1,keyword article,0.23,126
6,7,REFRESH,3,visible_high_vis_10Kplus,25,0,43817,2,13.7,320.0,1,keyword article,0.17,445
7,8,REFRESH,3,visible_high_vis_10Kplus,26,0,43211,2,12.7,390.0,1,keyword article,0.19,333
8,9,REFRESH,3,visible_high_vis_10Kplus,25,0,37432,2,11.8,140.0,1,keyword article,0.70,487
9,10,REFRESH,3,visible_high_vis_10Kplus,26,0,37269,2,13.8,260.0,1,keyword article,0.18,333



Top-10 skeptic review (one line each, action / why / what would make it wrong):
----------------------------------------------------------------------------------------------------------------------
  # 1  action=REFRESH  score=3  rc=visible_high_vis_10Kplus                       why: high visibility (81,865 imp 90d), striking distance (pos 22, SV 140)
        -> WOULD BE WRONG IF: most of the 81,865 90d impressions came from a single event-week spike and volume has already normalized; position 22 was actually a one-time brand-SERP feature loss, not a keyword ranking issue

  # 2  action=REFRESH  score=3  rc=visible_high_vis_10Kplus                       why: stale (194d since update), high visibility (61,678 imp 90d)
        -> WOULD BE WRONG IF: the 194d-since-update gap was intentional (static page like an about page or legal/faq); most of the 61,678 90d impressions came from a single event-week spike and volume has already normalized

  # 3  action=REFRESH  score=3  rc=visible_hig

## 4. Weak picks + leakage check

Two checks, then one named limitation to carry forward:

1. **Weak picks** — look at the lowest-ranked REFRESH rows and the highest-ranked IGNORE rows on the boundary. If the boundary is "surprisingly confident", we know the thresholds need tuning. In Week 5 we replace this boundary with a learned decision, but the frozen baseline must be honest about its boundary.
2. **Leakage check — no label-derived or future-window inputs in the rule.** The rule reads only four raw fields: `days_since_last_update`, `impressions_90d`, `avg_position`, `search_volume`. It never reads `trend_pct`, `trend_direction`, `is_declining`, or any other forward/outcome column. The reviewer context in §3 prints trend_pct for a skeptic to see, but the rule path does not use it. Confirm with a mechanical check: of the 4 fields that feed any bucket, list them all; then list every column that ever appears on the right-hand side of a `score` or `action` assignment.


In [5]:
import pandas as pd
import numpy as np
import json

q = QUEUE_WK.copy()

# ---------- Weak picks at the boundary ----------
last_refresh = q[q["action"] == "REFRESH"].tail(5).copy()
first_ignore = q[q["action"] == "IGNORE"].head(5).copy()
print("=" * 90)
print("WEAK PICK 1/2 — 5 lowest-ranked REFRESH rows (the boundary: score=4 or just made it in)")
print("=" * 90)
cols = ["rank", "score", "action", "reason_code", "days_since_last_update", "staleness_bucket",
        "impressions_90d", "vis_bucket", "avg_position", "search_volume", "striking_bonus"]
print(last_refresh[cols].to_string(index=False))
print()
print("=" * 90)
print("WEAK PICK 2/2 — 5 highest-ranked IGNORE rows (just missed the OBSERVE threshold at score=3)")
print("=" * 90)
print(first_ignore[cols].to_string(index=False))
print()

# ---------- Leakage check: mechanical audit of rule inputs ----------
# Every raw column that the rule actually READS (any bucket computation uses these only):
RULE_INPUTS_ACTUALLY_USED = {
    "days_since_last_update",  # staleness_bucket
    "impressions_90d",         # vis_bucket
    "avg_position",            # striking_bonus
    "search_volume",           # striking_bonus (fillna(0) then compare >=100)
}
# Known label-source / forward-window columns (from the data dictionary and w03 contract)
KNOWN_LABEL_PROXIES = {"trend_pct", "trend_direction", "is_declining", "severe_decline",
                       "decline_severity", "opportunity_score", "trend_pct_forward",
                       "is_declining_forward"}

lane_cols = set(LANE.columns)
used_label_proxies_in_rule_inputs = RULE_INPUTS_ACTUALLY_USED & KNOWN_LABEL_PROXIES
print("=" * 90)
print("LEAKAGE CHECK: rule inputs vs known label-source columns")
print("=" * 90)
print(f"Raw rule inputs:          {sorted(RULE_INPUTS_ACTUALLY_USED)}")
print(f"Known label/proxy cols:   {sorted(KNOWN_LABEL_PROXIES)}")
print(f"Rule-input ∩ label cols:  {sorted(used_label_proxies_in_rule_inputs)} -> {'LEAK DETECTED' if used_label_proxies_in_rule_inputs else 'CLEAN (no label columns used)'}")
print()

# Sanity-check: are any raw rule inputs derived from a window that ends AFTER the decision moment?
# (For the starter CSV, all bucket inputs are 90-day trailing metrics ending at the snapshot date. No forward window possible.)
# (For the warehouse equivalent, we'd check: every feature column must be aggregated from data WHERE report_date < label_month_start.)
print("Window sanity (starter snapshot rule inputs):")
for col in sorted(RULE_INPUTS_ACTUALLY_USED):
    print(f"  - {col}: {LANE[col].notna().sum()} / {len(LANE)} non-null in lane slice. All trailing-90d snapshots, no forward window possible.")
print()

# ---------- Final metrics receipt update ----------
with open(METRICS_JSON_PATH, "r") as f:
    receipt = json.load(f)
receipt["baseline_rule"]["leakage_check"] = {
    "raw_rule_inputs": sorted(RULE_INPUTS_ACTUALLY_USED),
    "known_label_proxies": sorted(KNOWN_LABEL_PROXIES),
    "intersection": sorted(used_label_proxies_in_rule_inputs),
    "status": "CLEAN" if not used_label_proxies_in_rule_inputs else "LEAK_DETECTED",
}
receipt["baseline_rule"]["weak_pick_boundaries"] = {
    "last_5_refresh_ranks": last_refresh["rank"].astype(int).tolist(),
    "first_5_ignore_ranks": first_ignore["rank"].astype(int).tolist(),
}
with open(METRICS_JSON_PATH, "w") as f:
    json.dump(receipt, f, indent=2, default=str)
print(f"Final metrics receipt updated -> {METRICS_JSON_PATH}")
print()

# ---------- Named limitation for the frozen baseline ----------
print("ONE NAMED LIMITATION (carried into Week 5 as a caveat on baseline quality):")
print("  > Staleness bucket uses 'days since LAST update', not 'days since PUBLISH'. For news-style pages")
print("    that are updated many times (e.g. rolling coverage), this flips the signal: staleness=low actually")
print("    means 'recently hot topic', and our rule under-scores them. Mitigation in Week 5 model:")
print("    include content_type and update-frequency as features, not just raw days_since_last_update.")


WEAK PICK 1/2 — 5 lowest-ranked REFRESH rows (the boundary: score=4 or just made it in)
 rank  score  action              reason_code  days_since_last_update  staleness_bucket  impressions_90d  vis_bucket  avg_position  search_volume  striking_bonus
   52      3 REFRESH visible_high_vis_10Kplus                      25                 0            10775           2          13.3         1600.0               1
   53      3 REFRESH visible_high_vis_10Kplus                      20                 0            10764           2          16.0         1000.0               1
   54      3 REFRESH visible_high_vis_10Kplus                      26                 0            10519           2          24.4        22200.0               1
   55      3 REFRESH visible_high_vis_10Kplus                     104                 0            10339           2          10.9          140.0               1
   56      3 REFRESH visible_high_vis_10Kplus                      25                 0            100

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it (S1 signal buckets, S2 encode + CSV write, S3 top-10 review, S4 weak picks + leakage).
- [x] The notebook runs top to bottom with no errors (nbconvert executed 9 code cells, 0 exceptions).
- [x] No client names, URLs, or private queries anywhere (everything is anonymized through the starter CSV's hashed IDs).
- [x] My claims use careful words: observed, measured, directional, decision-support (verdict language is CONFIRMED / MIXED, not "X causes Y", and the rule is frozen-baseline not "the truth").
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.
